# 2 · Von der MIDI-Datei zur neuen Melodie

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/02_von_midi_zur_melodie.ipynb)

Jetzt sind die Symbole **Noten**. Der Plan:

1. eine MIDI-Datei auswählen (eigene hochladen oder die Demo nehmen)
2. GreasyPidgin liest sie ein
3. lernen, welche Note auf welche zu folgen pflegt
4. eine neue Melodie erzeugen
5. sie in eine MIDI-Datei schreiben, die du herunterladen kannst

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

## Schritt 1: eine MIDI-Datei wählen

In Colab erscheint ein Knopf *Dateien auswählen*. Wähle eine `.mid`-Datei von deinem Computer.
Einstimmige Melodien funktionieren am besten (eine Gesangsstimme, eine Flötenstimme, eine Basslinie).
Beginnen mehrere Noten gleichzeitig, bleibt nur die höchste erhalten.

**Keine Datei zur Hand?** Brich den Dialog ab, dann wird die Demo-Melodie verwendet: *Freude, schöner Götterfunken*,
*Bruder Jakob* und *Morgen kommt der Weihnachtsmann* (die Melodie von *Twinkle, Twinkle*) hintereinander — so kann
die Kette die drei Stücke mischen.

In [ ]:
midi_path = wh.choose_midi()

## Schritt 2: mit GreasyPidgin einlesen

Im Hintergrund verwandelt `Gesture.fromMidiFile` jede MIDI-Note in ein **Phonon** — das Notenobjekt von
GreasyPidgin — und sammelt sie in einer **Gesture**. Wir holen Tonhöhen und Rhythmen heraus und
bewahren sie in einem kleinen `Melody`-Objekt auf.

In [ ]:
melody = wh.load_melody(midi_path)
print(melody)
wh.plot_melody(melody)
plt.show()

In [ ]:
wh.listen(midi_path)

`melody.pitches` sind MIDI-Notennummern (60 = mittleres C, geschrieben `c4`; `s` heißt Kreuz und `f` heißt b, also ist `cs4` = Cis4 und `bf3` = B3; deutsches H ist `b`), `melody.deltas` sind die Zeiten
zwischen den Notenanfängen in Schlägen (1.0 = eine Viertelnote), `melody.bpm` ist das Tempo.

In [ ]:
print(melody.pitches[:12])
print(melody.deltas[:12])
print(melody.bpm)

## Schritt 3: die Kette lernen

`ORDER` ist das Gedächtnis: wie viele vorherige Noten die nächste bestimmen.

In [ ]:
ORDER = 2

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=ORDER)
print(f"{len(pitch_chain.rules)} Regeln gelernt. Die ersten drei:")

for state in pitch_chain.rules.elements[:3]:
    names = " ".join(wh.pitch_name(p) for p in state)
    print(f"  nach  {names:8s} ->", pitch_chain.rules[state])

## Schritt 4: erzeugen

Wir beginnen die neue Melodie mit den ersten Noten des Originals und lassen die Kette weiterschreiben.

In [ ]:
N_NOTES = 32

start = tuple(melody.pitches[:ORDER])
new_pitches = pitch_chain.getNElements(seed=start, n=N_NOTES)

print(" ".join(wh.pitch_name(p) for p in new_pitches))

## Schritt 5: ein Rhythmus und eine MIDI-Datei

Vorerst borgt sich die neue Melodie den Rhythmus des Originals (und wiederholt ihn so oft wie nötig).
Im nächsten Kapitel lernt die Kette auch den Rhythmus.

Wir bauen eine **Gesture** aus parallelen Listen: eine Liste für die Tonhöhen, eine für die Zeit
zwischen den Einsätzen, eine für die Dauern, dazu eine Dynamik und ein Tempo.

In [ ]:
from GreasyPidgin import Gesture

rhythm = [melody.deltas[i % len(melody.deltas)] for i in range(len(new_pitches))]

gesture = Gesture.fromLists(
    size        = len(new_pitches),
    startOffset = 0,
    startDeltas = [0] + rhythm[:-1],           # Zeit vor dem Einsatz jeder Note
    durations   = [r * 0.9 for r in rhythm],   # leicht abgesetzt
    pitches     = new_pitches,
    dynamics    = 80,                          # MIDI-Velocity
    timeUnit    = "b",                         # alle Zeiten in Schlägen
    bpm         = melody.bpm,
)

gesture.toMidi("new_melody.mid", bpm=melody.bpm)

In [ ]:
wh.plot_notes(new_pitches, rhythm, title="Die neue Melodie", color="C1")
plt.show()
wh.listen("new_melody.mid")

Gefällt sie dir? Dann speichere die MIDI-Datei auf deinem Computer:

In [ ]:
wh.download("new_melody.mid")

## Nochmal, nochmal!

Geh zurück zu **Schritt 4** und führe die Zellen darunter erneut aus: jeder Durchlauf ergibt eine neue Melodie.

**Probier es aus:**

- Ändere `ORDER` in Schritt 3 auf `1` oder `3` (und führe Schritt 3 und 4 erneut aus). Klingt die neue Melodie
  noch wie das Original?
- Mach sie länger (`N_NOTES = 100`).
- Lade eine andere MIDI-Datei hoch.
- Ändere die `0.9` in Schritt 5 auf `1.0` (legato) oder `0.4` (staccato).